# pl-cities-lakehouse on Microsoft Fabric

The same pipeline as the Databricks job (`notebooks/01`–`03`), in one Fabric notebook, for the
Databricks vs Fabric comparison. The logic comes from the project wheel (`pl_cities_lakehouse`),
so not a single transformation is rewritten.

> **Run on 2026-11-18** (Fabric trial, Poland Central): first run 4 min 26 s, incremental run 1 min 37 s.
> Results and lessons: `docs/databricks-vs-fabric.md`.

- **Files** of the default lakehouse play the role of the Unity Catalog volume (raw files by load date).
- **Tables** in schemas `bronze`, `silver`, `gold` are Delta tables, like `pl_cities.*` on Databricks.
- Every stage is timed; the last cell prints the times for the comparison.

Before the first run: attach the lakehouse `pl_cities` (created with *Lakehouse schemas* on) and upload
the wheel to `Files/libs/`.

In [ ]:
# -I (ignore installed) reinstalls the wheel after it is uploaded again with new code.
%pip install openpyxl -q
%pip install /lakehouse/default/Files/libs/pl_cities_lakehouse-0.1.0-py3-none-any.whl --no-deps -Iq

In [ ]:
import time
from contextlib import contextmanager
from datetime import date
from pathlib import Path

from pl_cities_lakehouse.common.cities import CITIES
from pl_cities_lakehouse.common.delta import merge_into
from pl_cities_lakehouse.common.teams import STADIUM_TOWNS, TEAM_TOWNS
from pl_cities_lakehouse.ingestion import football, gios, gus_bdl, nbp_housing, open_meteo
from pl_cities_lakehouse.pipeline import gold_tables, silver_tables

RAW = Path("/lakehouse/default/Files/raw")  # default lakehouse, mounted as a local folder
TODAY = date.today().isoformat()
towns = {**CITIES, **STADIUM_TOWNS}
cities = list(CITIES)
timings = {}


@contextmanager
def step(name):
    """Time one stage of the pipeline for the comparison with Databricks."""
    start = time.perf_counter()
    yield
    timings[name] = round(time.perf_counter() - start, 1)
    print(f"--- {name}: {timings[name]} s")


def save(df, table):
    """Full overwrite of a Delta table; prints the row count as a quick check."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table:30} {spark.table(table).count():>8} rows")


for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

In [ ]:
# Bronze weather: full history on the first run, then the missing days + last week with MERGE.
with step("bronze_weather"):
    table = "bronze.weather_daily"
    if not spark.catalog.tableExists(table):
        open_meteo.save_raw_responses(towns, RAW / "open_meteo")
        records = open_meteo.read_records(towns, RAW / "open_meteo")
        open_meteo.to_dataframe(spark, records).write.saveAsTable(table)
        print("created", table)
    loaded = open_meteo.last_dates(spark.table(table))
    ranges = open_meteo.incremental_ranges(towns, loaded, date.today())
    saved = open_meteo.save_increments(towns, ranges, RAW / "open_meteo" / "increments")
    records = open_meteo.read_increment_records(saved)
    if records:
        merge_into(open_meteo.to_dataframe(spark, records), table, ["city", "date"])
    print(len(records), "rows merged into", table)

In [ ]:
# Bronze snapshots: each source publishes its whole history, so a fresh copy and a full overwrite.
with step("bronze_snapshots"):
    nbp_file = nbp_housing.download(RAW / "nbp" / TODAY / "ceny_mieszkan.xlsx")
    save(
        nbp_housing.to_dataframe(spark, nbp_housing.read_records(nbp_file), nbp_file.name),
        "bronze.housing_prices",
    )
    wages_file = gus_bdl.save_raw(
        gus_bdl.fetch_variable(gus_bdl.WAGE_VARIABLE_ID),
        RAW / "gus_bdl" / TODAY / f"wages_{gus_bdl.WAGE_VARIABLE_ID}.json",
    )
    save(
        gus_bdl.to_dataframe(spark, gus_bdl.to_records(gus_bdl.load_raw(wages_file), cities)),
        "bronze.wages",
    )
    gios.download(RAW / "gios")
    save(gios.to_dataframe(spark, gios.read_records(RAW / "gios", cities)), "bronze.pm10")
    matches_file = football.download(RAW / "football" / TODAY / "POL.csv")
    save(
        football.to_dataframe(
            spark, football.read_records(matches_file, TEAM_TOWNS), matches_file.name
        ),
        "bronze.matches",
    )

In [ ]:
# Silver and gold: the same functions as on Databricks and locally.
NAMES = ["weather_daily", "housing_prices", "wages", "pm10", "matches"]

with step("silver"):
    bronze = {name: spark.table(f"bronze.{name}") for name in NAMES}
    for name, df in silver_tables(bronze).items():
        save(df, f"silver.{name}")

with step("gold"):
    silver = {name: spark.table(f"silver.{name}") for name in NAMES}
    for name, df in gold_tables(spark, silver).items():
        save(df, f"gold.{name}")

In [ ]:
# Times for docs/databricks-vs-fabric.md and the same ranking check as on Databricks.
for name, seconds in timings.items():
    print(f"{name:20} {seconds:>7} s")
print(f"{'total':20} {round(sum(timings.values()), 1):>7} s")

display(spark.sql("""
    SELECT market, final_rank, city, ROUND(final_score, 2) AS final_score
    FROM gold.mart_city_score
    WHERE year = 2025 AND final_rank <= 3
    ORDER BY market, final_rank
"""))